# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
import NCA

using KadanoffBaym
include("../src/wigner.jl") # Wigner transform

import Parameters: @unpack
import SpecialFunctions: erf, erfc
import LinearAlgebra: diag

using Plots
default(fmt = :png)

## Initial correlations

### Equilibrium state

In [ ]:
# Solve the problem in equilibrium
model = NCA.SIAM(
  V₀=sqrt(0.2),
  ε₀f=-0.5)

nca = NCA.FreqNCA(
  model=model,
  β=5e2,
  mesh=NCA.LinearMesh(-3.0,3.0,2^13,false),
  eps=1e-9)

imΣ_t = NCA.solve!(nca; project=false)

In [ ]:
# Calculate Green functions
GᶜA, GᶠA, GᵇA = begin
  Aᶜ = nca.data.A_c
  Gᶜ = NCA.kramerskronig(nca.mesh, π * Aᶜ, :real) + 1im * π * Aᶜ;
    
  G = NCA.slave_particle_gf(nca, [nca.data.fermi_neg.*imΣ_t[1], nca.data.fermi_neg.*imΣ_t[2]])
  Gᵇ = G[1]
  Gᶠ = G[2]
    
  imGᵇ = view(reinterpret(Float64, Gᵇ),2:2:length(Gᵇ))
  imGᵇ[imGᵇ .< nca.eps] .= 0.0
    
  imGᶠ = view(reinterpret(Float64, Gᶠ),2:2:length(Gᶠ))
  imGᶠ[imGᶠ .< nca.eps] .= 0.0
    
  Gᶜ, Gᶠ, Gᵇ
end

# Allow for complex convolutions
mesh = NCA.LinearMesh(nca.mesh.xmin, nca.mesh.xmax, nca.mesh.L, true);
∫dω(f) = trapz(mesh, f) / (2π)

In [ ]:
# Assert GFs are normalized!
println(∫dω(GᶜA - conj(GᶜA)) |> imag)
println(∫dω(GᶠA - conj(GᶠA)) |> imag)
println(∫dω(GᵇA - conj(GᵇA)) |> imag)

# plot(mesh.mesh, GᵇA |> imag, label=L"$A_b$", title="Spectral functions", xlab=L"$\omega$")
# plot!(mesh.mesh, GᶠA |> imag, label=L"$A_f$")
# plot!(mesh.mesh, GᶜA |> imag, label=L"$A_c$")

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
# Gaussian pulse parameters
const tc = 40.  # peak arrival time
const Ω = 0.12 # frequency bandwidth of the pulse

function ξ(t)
    return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

# Pump and loss terms
const Γ = 5.
λ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t) / sqrt(0.5 * erfc((t - tc) * Ω / sqrt(2)))
γ(t::Float64) = iszero(Γ) ? 0.0 : Γ * ξ(t)

# Photon field Hamiltonian
h(t::Float64) = 1e-2 - 0.5im * (λ(t) + γ(t))

# Artificial width of the peak
Σᵞ = 1e-2

# Initial field occupation
n₀ = 0.

## Initial conditions

In [ ]:
u0_vert = begin
  Aᶜ = 2 * imag(GᶜA)
  Aᶠ = 2 * imag(GᶠA)
  Aᵇ = 2 * imag(GᵇA)

  # Generate grids
  bose = NCA.nb.(nca.mesh.mesh, nca.β)
  fermi = NCA.nf.(nca.mesh.mesh, nca.β);

  G₀ᶜL = +1.0im * ∫dω(fermi .* Aᶜ)
  G₀ᶜL = GreenFunction(G₀ᶜL * ones(1,1), Lesser)

  G₀ᶜG = -1.0im * (1 - ∫dω(fermi .* Aᶜ))
  G₀ᶜG = GreenFunction(G₀ᶜG * ones(1,1), Greater)

  G₀ᶠL = +1.0im * ∫dω(fermi .* Aᶠ)
  G₀ᶠL = GreenFunction(G₀ᶠL * ones(1,1), Lesser)

  G₀ᶠG = -1.0im * (1 - ∫dω(fermi .* Aᶠ))
  G₀ᶠG = GreenFunction(G₀ᶠG * ones(1,1), Greater)

  G₀ᵇL = -1.0im * ∫dω(bose .* Aᵇ)
  G₀ᵇL = GreenFunction(G₀ᵇL * ones(1,1), Lesser)

  G₀ᵇG = -1.0im * (1 + ∫dω(bose .* Aᵇ))
  G₀ᵇG = GreenFunction(G₀ᵇG * ones(1,1), Greater)

  G₀ᵞL = -1.0im * n₀
  G₀ᵞL = GreenFunction(G₀ᵞL * ones(1,1), Lesser)

  G₀ᵞG = -1.0im * (1 + n₀)
  G₀ᵞG = GreenFunction(G₀ᵞG * ones(1,1), Greater)

 [G₀ᶜL, G₀ᶜG, G₀ᶠL, G₀ᶠG, G₀ᵇL, G₀ᵇG, G₀ᵞL, G₀ᵞG];
end

u0_line = [[GᶜA], [conj(GᶜA)], [GᶠA], [conj(GᶠA)], [GᵇA], [conj(GᵇA)]]

u0 = [u0_vert, u0_line];

### ODEs

$$
i \partial_t G^\gtrless(t,t^\prime) =
h(t) G^\gtrless(t,t^\prime)
+ \int_{t_0}^t \textrm{d}\bar{t} \left[\Sigma^>(t,\bar{t}) - \Sigma^<(t,\bar{t})\right] G^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} \Sigma^\gtrless(t, \bar{t}) \left[G^>(\bar{t},t^\prime) - G^<(\bar{t},t^\prime)\right]
- i \int_0^\beta \textrm{d}\tau \Sigma^\rceil(t,\tau) G^\lceil(\tau,t^\prime)
$$
$$
-i \partial_{t^\prime} G^\gtrless(t,t^\prime) =
G^\gtrless(t,t^\prime) h(t)^\dagger
+ \int_{t_0}^{t} \textrm{d}\bar{t} \left[G^>(t,\bar{t}) - G^<(t,\bar{t})\right] \Sigma^\gtrless(\bar{t},t^\prime)
- \int_{t_0}^{t^\prime} \textrm{d}\bar{t} G^\gtrless(t,\bar{t}) \left[\Sigma^>(\bar{t},t^\prime) - \Sigma^<(\bar{t},t^\prime)\right] 
- i \int_0^\beta \textrm{d}\tau G^\rceil(t,\tau)\Sigma^\lceil(\tau,t^\prime) 
$$

In [ ]:
using Parameters

σ = 2.0 # Conduction electron width
Γ₀ = 0.2 # Light-matter coupling
V₀ = model.V₀ # Hybridization strength

# Define commutator
⋊(a,b) = a * b - b * adjoint(a)

# maps a G˥ to a G⌐
⌐(A) = conj(reverse(A)) # NOTE: underlying grid has to be symmetric!

struct SelfEnergies
    ΣᶠL::GreenFunction
    ΣᶠG::GreenFunction
    ΣᵇL::GreenFunction
    ΣᵇG::GreenFunction
    
    Σᶠ˥p::Vector{Vector{ComplexF64}}
    Σᶠ˥m::Vector{Vector{ComplexF64}}
    Σᵇ˥p::Vector{Vector{ComplexF64}}
    Σᵇ˥m::Vector{Vector{ComplexF64}}
    
    function SelfEnergies()
        new(GreenFunction(zeros(ComplexF64,0,0), Lesser),
            GreenFunction(zeros(ComplexF64,0,0), Greater),
            GreenFunction(zeros(ComplexF64,0,0), Lesser),
            GreenFunction(zeros(ComplexF64,0,0), Greater),
            [[]],
            [[]],
            [[]],
            [[]])
    end
end

function update_2time!(Σs::SelfEnergies, u::AbstractArray, i, j)
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG = Σs
    
    if size(ΣᶠL,1) < max(i,j)
        Tmax = size(first(u),1)
        
        resize!(ΣᶠL,Tmax)
        resize!(ΣᶠG,Tmax)
        resize!(ΣᵇL,Tmax)
        resize!(ΣᵇG,Tmax)
    end
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u
    
    ΣᶠL[i,j] = GᶜL[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᵇL[i,j]
    ΣᶠG[i,j] = GᶜG[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᵇG[i,j]
    ΣᵇL[i,j] = 2 * GᶠL[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᶜG[j,i]
    ΣᵇG[i,j] = 2 * GᶠG[i,j] * (V₀^2 + Γ₀^2 * (GᵞL[i,j] + GᵞG[j,i])) * GᶜL[j,i]
end

function update_1time!(Σs::SelfEnergies, u::AbstractArray, v::AbstractArray, i)
    @unpack Σᶠ˥p, Σᶠ˥m, Σᵇ˥p, Σᵇ˥m = Σs
    
    update_2time!(Σs::SelfEnergies, u::AbstractArray, i, 1)
    
    if size(Σᶠ˥p,1) < i
        Tmax = size(first(u),1)
        resize!(Σᶠ˥p,Tmax)
        resize!(Σᶠ˥m,Tmax)
        resize!(Σᵇ˥p,Tmax)
        resize!(Σᵇ˥m,Tmax)
    end
    
    # unpack Green functions
    Gᶜ˥m, Gᶜ˥p, Gᶠ˥m, Gᶠ˥p, Gᵇ˥m, Gᵇ˥p = v
    
    Aᶜ˥ = (Gᶜ˥p[i] - Gᶜ˥m[i])
    Aᶜ˥p = nca.data.fermi .* Aᶜ˥
    Aᶜ˥h = nca.data.fermi_neg .* Aᶜ˥
    
    Σᶠ˥m[i] = -1 * abs2(V₀) * NCA.conv(mesh, Aᶜ˥h, Gᵇ˥m[i]) / (2π)
    Σᶠ˥p[i] = -1 * abs2(V₀) * NCA.conv(mesh, Aᶜ˥h, Gᵇ˥p[i]) / (2π)
    Σᵇ˥m[i] = -2 * abs2(V₀) * NCA.xcorr(mesh, Aᶜ˥p, Gᶠ˥m[i]) / (2π)
    Σᵇ˥p[i] = -2 * abs2(V₀) * NCA.xcorr(mesh, Aᶜ˥p, Gᶠ˥p[i]) / (2π)
end

Σs = SelfEnergies();

In [ ]:
"""
Vertical stepping
"""
f_vert(u,v,times,t1,t2) = f_vert_2time(u,times,t1,t2) .+ f_vert_freq(v,times,t1,t2)
f_diag(u,v,times,t1) = f_diag_2time(u,times,t1) .+ f_diag_freq(v,times,t1)

function f_vert_2time(u, times, t1, t2)
    # Real-time collision integral 
    ∫dt(i,j,A,B) = sign(j-i) * trapz(times[min(i,j):max(i,j)], [A[t1,t] * B[t,t2] for t=min(i,j):max(i,j)])
    
    update_2time!(Σs, u, t1, t2)
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG = Σs
    
    # conduction electron
    dGᶜG = -1.0im * (-1.0im * σ * GᶜG[t1,t2])
    dGᶜL = -1.0im * (-1.0im * σ * GᶜL[t1,t2])
    
    # pseudo fermion
    dGᶠG = -1.0im * (model.ε₀f * GᶠG[t1,t2] + ∫dt(t2,t1,ΣᶠG,GᶠG))
    dGᶠL = -1.0im * (model.ε₀f * GᶠL[t1,t2] + ∫dt(1,t1,ΣᶠG,GᶠL) - ∫dt(1,t2,ΣᶠL,GᶠG))
     
    # slave boson
    dGᵇG = -1.0im * (zero(GᵇG[t1,t2]) + ∫dt(t2,t1,ΣᵇG,GᵇG))
    dGᵇL = -1.0im * (zero(GᵇL[t1,t2]) + ∫dt(1,t1,ΣᵇG,GᵇL) - ∫dt(1,t2,ΣᵇL,GᵇG))

    # thz pulse
    dGᵞL = -1.0im * ((h(times[t1]) + 1.0im * γ(times[t1]) - 1.0im * Σᵞ) * GᵞL[t1,t2])
    dGᵞG = -1.0im * ((h(times[t1]) + 1.0im * γ(times[t1]) - 1.0im * Σᵞ) * GᵞG[t1,t2])
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

function f_vert_freq(v, times, t1, t2)
    Gᶜ˥m, Gᶜ˥p, Gᶠ˥m, Gᶠ˥p, Gᵇ˥m, Gᵇ˥p = v
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG, Σᶠ˥p, Σᶠ˥m, Σᵇ˥p, Σᵇ˥m = Σs
    
    # conduction electron
    dGᶜG = -1.0im * (zero(ΣᶠG[t1,t2]))
    dGᶜL = -1.0im * (zero(ΣᶠG[t1,t2]))
    
    # pseudo fermion
    dGᶠG = -1.0im * (zero(ΣᶠG[t1,t2]))
    dGᶠL = -1.0im * (∫dω(fermi .* (Σᶠ˥m[t1] .* Gᶠ˥m[t2] - Σᶠ˥p[t1] .* Gᶠ˥p[t2])))
#     dGᶠL = -1.0im * (-∫dω((1 .- fermi) .* (Σᶠ˥m[t1] .* ⌐(Gᶠ˥p[t2]) - Σᶠ˥p[t1] .* ⌐(Gᶠ˥m[t2]))))
    
    # slave boson
    dGᵇG = -1.0im * (zero(ΣᵇG[t1,t2]))
    dGᵇL = -1.0im * (-∫dω(bose .* (Σᵇ˥m[t1] .* Gᵇ˥m[t2] - Σᵇ˥p[t1] .* Gᵇ˥p[t2])))
#     dGᵇL = -1.0im * (-∫dω((1 .+ bose) .* (Σᵇ˥m[t1] .* ⌐(Gᵇ˥p[t2]) - Σᵇ˥p[t1] .* ⌐(Gᵇ˥m[t2]))))
    
    # thz pulse
    dGᵞL = -1.0im * (zero(ΣᶠG[t1,t2]))
    dGᵞG = -1.0im * (zero(ΣᶠG[t1,t2]))
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

function f_diag_2time(u, times, t1)  
    println("t: $(times[t1])")
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG = Σs
    
    # conduction electron
    dGᶜG = -1.0im * (zero(GᶜG[t1,t1]))
    dGᶜL = -1.0im * (zero(GᶜL[t1,t1]))

    # pseudo fermion
    dGᶠG = -1.0im * (zero(GᶠG[t1,t1]))
    dGᶠL = -1.0im * (zero(GᶠL[t1,t1]))
    
    # slave boson
    dGᵇG = -1.0im * (zero(GᵇG[t1,t1]))
    dGᵇL = -1.0im * (zero(GᵇL[t1,t1]))
    
    # thz pulse
    dGᵞL = -1.0im * (h(times[t1]) ⋊ GᵞL[t1,t1] + 1.0im * γ(times[t1]) * (GᵞL[t1,t1] + GᵞG[t1,t1]))
    dGᵞG = -1.0im * (h(times[t1]) ⋊ GᵞG[t1,t1] + 1.0im * λ(times[t1]) * (GᵞG[t1,t1] - GᵞL[t1,t1]) 
        + 2.0im * γ(times[t1]) * GᵞG[t1,t1])
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

function f_diag_freq(v, times, t1)
    # unpack Green functions
    Gᶜ˥m, Gᶜ˥p, Gᶠ˥m, Gᶠ˥p, Gᵇ˥m, Gᵇ˥p = v
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG, Σᶠ˥p, Σᶠ˥m, Σᵇ˥p, Σᵇ˥m = Σs
    
    # conduction electron
    dGᶜG = 0.0
    dGᶜL = 0.0

    # pseudo fermion
    val = -1.0im * (
        +∫dω(fermi .* (Σᶠ˥m[t1] .* Gᶠ˥m[t1] - Σᶠ˥p[t1] .* Gᶠ˥p[t1]))
        +∫dω((1 .- fermi) .* (Gᶠ˥m[t1] .* Σᶠ˥m[t1] - Gᶠ˥p[t1] .* Σᶠ˥p[t1])))
    dGᶠG = val
    dGᶠL = val
    
    # slave boson
    val = -1.0im * (
        -∫dω(bose .* (Σᵇ˥m[t1] .* Gᵇ˥m[t1] - Σᵇ˥p[t1] .* Gᵇ˥p[t1]))
        +∫dω((1 .+ bose) .* (Gᵇ˥m[t1] .* Σᵇ˥m[t1] - Gᵇ˥p[t1] .* Σᵇ˥p[t1])))
    dGᵇG = val
    dGᵇL = val
    
    # thz pulse
    dGᵞL = 0.0
    dGᵞG = 0.0
    
    return [dGᶜL, dGᶜG, dGᶠL, dGᶠG, dGᵇL, dGᵇG, dGᵞL, dGᵞG]
end

function f_line(u, v, times, t1)
    # Integral 
    ∫dt(A,B) = trapz(times, [A[t1,t] * B[t] for t=1:t1])
    
    update_1time!(Σs, u, v, t1)
    
    # unpack Green functions
    GᶜL, GᶜG, GᶠL, GᶠG, GᵇL, GᵇG, GᵞL, GᵞG = u
    Gᶜ˥m, Gᶜ˥p, Gᶠ˥m, Gᶠ˥p, Gᵇ˥m, Gᵇ˥p = v
    @unpack ΣᶠL, ΣᶠG, ΣᵇL, ΣᵇG, Σᶠ˥p, Σᶠ˥m, Σᵇ˥p, Σᵇ˥m = Σs

    # conduction electron
    dGᶜ˥p = -1.0im * (-1.0im * σ * Gᶜ˥p[t1])
    dGᶜ˥m = -1.0im * (-1.0im * σ * Gᶜ˥m[t1])

    # pseudo fermion
    dGᶠ˥p = -1.0im * (model.ε₀f * Gᶠ˥p[t1] + ∫dt(ΣᶠG, Gᶠ˥p) + Σᶠ˥p[t1] .* Gᶠ˥p[1])
    dGᶠ˥m = -1.0im * (model.ε₀f * Gᶠ˥m[t1] + ∫dt(ΣᶠG, Gᶠ˥m) + Σᶠ˥m[t1] .* Gᶠ˥m[1])
    
    # slave boson
    dGᵇ˥p = -1.0im * (zero(Gᵇ˥p[t1]) + ∫dt(ΣᵇG, Gᵇ˥p) + Σᵇ˥p[t1] .* Gᵇ˥p[1])
    dGᵇ˥m = -1.0im * (zero(Gᵇ˥m[t1]) + ∫dt(ΣᵇG, Gᵇ˥m) + Σᵇ˥m[t1] .* Gᵇ˥m[1])

    return [dGᶜ˥m, dGᶜ˥p, dGᶠ˥m, dGᶠ˥p, dGᵇ˥m, dGᵇ˥p]
end

In [ ]:
# KMS Boundary conditions

# G^R(t,t) = -i
@assert u0[1][2].data - u0[1][1].data ≈ -1.0im * ones(1,1)
@assert u0[1][4].data - u0[1][3].data ≈ -1.0im * ones(1,1)
@assert u0[1][6].data - u0[1][5].data ≈ -1.0im * ones(1,1)
@assert u0[1][8].data - u0[1][7].data ≈ -1.0im * ones(1,1)

# C^<(t,0) = C˥(t,0+)
@assert u0[1][1].data[1,1] ≈ ∫dω(fermi .* (u0[2][1][1] - u0[2][2][1]))
@assert u0[1][3].data[1,1] ≈ ∫dω(fermi .* (u0[2][3][1] - u0[2][4][1]))
@assert u0[1][5].data[1,1] ≈ -∫dω(bose .* (u0[2][5][1] - u0[2][6][1]))

In [ ]:
# ODE problem is defined by the rhs, initial condition and time span
tspan = (0.0,20)
u, times = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; f_line=f_line,
    dt=1e-3, max_dt=0.25,adaptive=true, atol=1e-9, rtol=1e-7)

In [ ]:
# KMS Boundary conditions
≅(a,b) = isapprox(a,b;atol=1e-3)

# G^R(t,t) = -i
@assert all(diag(u[1][2].data) - diag(u[1][1].data) .≅ -1.0im)
@assert all(diag(u[1][4].data) - diag(u[1][3].data) .≅ -1.0im)
@assert all(diag(u[1][6].data) - diag(u[1][5].data) .≅ -1.0im)
@assert all(diag(u[1][8].data) - diag(u[1][7].data) .≅ -1.0im)

# C^<(t,0) = C˥(t,0+)
@assert u[1][1].data[:,1] ≅ [∫dω(fermi .* (u[2][1][t] - u[2][2][t])) for t in eachindex(times)]
@assert u[1][3].data[:,1] ≅ [∫dω(fermi .* (u[2][3][t] - u[2][4][t])) for t in eachindex(times)]
@assert u[1][5].data[:,1] ≅ [-∫dω(bose .* (u[2][5][t] - u[2][6][t])) for t in eachindex(times)]

In [ ]:
# GᶜL = u[1]
# GᶜG = u[2]

# GᶠL = u[3]
# GᶠG = u[4]

# GᵇL = u[5]
# GᵇG = u[6]

# GᵞL = u[7];
# GᵞG = u[8];

GᶜL = u[1][1]
GᶜG = u[1][2]

GᶠL = u[1][3]
GᶠG = u[1][4]

GᵇL = u[1][5]
GᵇG = u[1][6]

GᵞL = u[1][7];
GᵞG = u[1][8];

In [ ]:
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
# heatmap(times, times, -imag(GᶜL.data))
# heatmap(times, times, -imag(GᶠL.data))
heatmap(times, times, -imag(GᵇL.data))
# heatmap(times, times, -imag(GᵞL.data))

In [ ]:
ρ = wigner_transform(GᵇG.data - GᵇL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᶠG.data - GᶠL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-10,10))

In [ ]:
ρ = wigner_transform(GᵞG.data - GᵞL.data)#; Nf=4096*2);
wigner_plot!(real(ρ), times)#; ylims=(-3,3))

In [ ]:
plot(u[2][5][40]|>imag)